# Creates the knowledgebase.txt file

This notebook creates the knowledgebase.txt file.  This file contains one line per 'data/cat-fact.txt' entry.

The knowledge base contains both the fact and the embedding model's semnatic vector for the fact.

# Install packages, if not present

In [1]:
!pip install ollama --quiet
!pip install import-ipynb --quiet

# Configuration values

Should I revisit and move these values into a configuration file.

In [11]:
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
FACT_DATABASE = 'data/cat-facts.txt'
KNOWLEDGE_BASE_FILE = 'knowledgebase.txt'

# Logging Control.

For more detailed operation output, set 

```
logging.basicConfig(level=logging.DEBUG)
```

instead of the default logging.INFO level.

In [3]:
import logging

logging.basicConfig(level=logging.INFO)

# we suppress logging for HTTP calls
logging.getLogger('httpx').setLevel(logging.ERROR)
logging.getLogger('httpcore').setLevel(logging.ERROR)

import_ipynb permits imports from other ipynb notebooks
- Warning: the loaded notebooks capture state.  Notably the logging state.  To fix this, restart the kernel and load each imported notebook.

The DocumentLoader retrieves facts (one for each line) from a fact file database 'data/cat-facts.txt'.

The SemanticLayer converts the facts to semantic encoded vectors.

The KnowledgeBase writes the facts to our file-based "vector database".

In [14]:
import import_ipynb

import logging

import DocumentLoader
import SemanticLayer
import KnowledgeBase

logger = logging.getLogger(__name__)

logger.info(f"creating the knowledge base.")

loader = DocumentLoader.DocumentLoader(filename=FACT_DATABASE)
semantic = SemanticLayer.SemanticLayer(model=EMBEDDING_MODEL)
knowledge = KnowledgeBase.KnowledgeBase(filename=KNOWLEDGE_BASE_FILE)

knowledge.reset()
for fact in loader:
    encoding = semantic.getEncoding(text=fact)
    knowledge.insert(text=fact, encoding=encoding)
loader.close()
knowledge.close()
logger.info(f"knowledge base build complete.")

INFO:__main__:creating the knowledge base.
INFO:DocumentLoader:opening file data/cat-facts.txt for reading.
INFO:SemanticLayer:pulling model hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
INFO:SemanticLayer:  pulling manifest
INFO:SemanticLayer:  pulling 74aebb552ea7
INFO:SemanticLayer:  pulling b1899e715228
INFO:SemanticLayer:  verifying sha256 digest
INFO:SemanticLayer:  writing manifest
INFO:SemanticLayer:  success
INFO:SemanticLayer:pull of model hf.co/CompendiumLabs/bge-base-en-v1.5-gguf complete
INFO:KnowledgeBase:opening knowledge base knowledgebase.txt.
INFO:DocumentLoader:no more facts in data/cat-facts.txt.
INFO:DocumentLoader:closing file data/cat-facts.txt.
INFO:KnowledgeBase:Closing knowledge base knowledgebase.txt.
INFO:__main__:knowledge base build complete.
